# Second Part: Data cleaning and dashboard preparation

This notebook converts the filtered CORDIS and CTIS datasets into stable, dashboard-ready tables. It standardizes data types, removes export artifacts, adds readable country names, and normalizes the many-country clinical-trial location field.

## Outputs

- `data_final/ConsurDatabase_clean.csv`: one row per project–organisation relationship.
- `data_final/TrialsDatabase_clean.csv`: one row per clinical trial.
- `data_final/TrialCountries.csv`: one row per unique trial–country relationship.


## 1. Imports and paths

In [1]:
import sys
sys.executable
import sys
!{sys.executable} -m pip install babel

In [2]:
from pathlib import Path
import re

import pandas as pd
from babel import Locale

In [3]:
consur_path = Path("../../_local_data_backup/ConsurDatabase.csv")
trials_path = Path("data/data_final/TrialsDatabase.csv")

df_consur = pd.read_csv(consur_path)
df_trials = pd.read_csv(trials_path)

/var/folders/_z/vq1q1pzj639b28jbhlxzd0zw0000gn/T/ipykernel_42773/1285446187.py:4: DtypeWarning: Columns (0: active) have mixed types. Specify dtype option on import or set low_memory=False.
  df_consur = pd.read_csv(consur_path)


## 2. Reusable cleaning functions

In [4]:
ENGLISH = Locale("en")
COUNTRY_OVERRIDES = {
    "EL": "Greece",      
    "UK": "United Kingdom",  
    "XK": "Kosovo",
    "ZZ": "Unknown",
}


def remove_export_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Remove index columns created by previous CSV exports."""
    return df.loc[:, ~df.columns.str.match(r"^Unnamed:")].copy()


def clean_text(series: pd.Series) -> pd.Series:
    """Trim text and collapse repeated whitespace while retaining nulls."""
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .replace("", pd.NA)
    )


def country_name_from_code(value):
    """Translate one or more CORDIS alpha-2 codes into English country names."""
    if pd.isna(value):
        return pd.NA
    codes = [code.strip().upper() for code in re.split(r"[;,]", str(value)) if code.strip()]
    names = []
    for code in codes:
        name = COUNTRY_OVERRIDES.get(code) or ENGLISH.territories.get(code)
        names.append(name or f"Unknown ({code})")
    return "; ".join(names) if names else pd.NA


def split_trial_locations(trials: pd.DataFrame) -> pd.DataFrame:
    """Create one unique trial–country row from CTIS location/status strings."""
    source_column = "Location(s)_and_recruitment_status"
    locations = trials[["Trial_number", source_column]].dropna().copy()
    locations["location_entry"] = locations[source_column].str.split(r"\s*,\s*")
    locations = locations.explode("location_entry", ignore_index=True)

    parts = locations["location_entry"].str.split(":", n=1, expand=True)
    locations["country_name"] = clean_text(parts[0])
    locations["recruitment_status"] = (
        clean_text(parts[1]) if parts.shape[1] > 1 else pd.NA
    )
    locations = locations.dropna(subset=["Trial_number", "country_name"])
    return locations[
        ["Trial_number", "country_name", "recruitment_status"]
    ].drop_duplicates().reset_index(drop=True)


## 3. Clean the CORDIS opportunity table


In [6]:
consur = remove_export_columns(df_consur)

text_columns = [
    "projectAcronym", "name", "shortName", "activityType", "city", "country",
    "role", "status", "title", "topics", "fundingScheme", "period",
]
for column in text_columns:
    if column in consur:
        consur[column] = clean_text(consur[column])

numeric_columns = [
    "ecContribution", "netEcContribution", "totalCostOrg", "totalCostProj",
]
for column in numeric_columns:
    if column in consur:
        consur[column] = pd.to_numeric(consur[column], errors="coerce")

for column in ["startDate", "endDate", "contentUpdateDate"]:
    if column in consur:
        consur[column] = pd.to_datetime(consur[column], errors="coerce")

consur["country_code"] = consur["country"].str.upper()
consur["country_name"] = consur["country_code"].map(country_name_from_code)
consur["project_start_year"] = consur["startDate"].dt.year.astype("Int64")
consur["project_end_year"] = consur["endDate"].dt.year.astype("Int64")
consur["organisation_project_key"] = (
    consur["organisationID"].astype("string") + "|" + consur["projectID"].astype("string")
)

before = len(consur)
consur = consur.drop_duplicates().reset_index(drop=True)
print(f"CORDIS rows: {before:,} -> {len(consur):,} ({before - len(consur):,} exact duplicates removed)")
print(f"Projects: {consur['projectID'].nunique():,}")
print(f"Countries: {consur['country_name'].nunique():,}")
consur[["country_code", "country_name"]].drop_duplicates().sort_values("country_name").head(15)

CORDIS rows: 201,690 -> 201,690 (0 exact duplicates removed)
Projects: 36,936
Countries: 190


,country_code,country_name
54917,AF,Afghanistan
12270,AL,Albania
6320,DZ,Algeria
162581,AD,Andorra
33451,AO,Angola
25657,AI,Anguilla
2745,AR,Argentina
5112,AM,Armenia
143217,AW,Aruba
163,AU,Australia


## 4. Clean and normalize the CTIS trial table

In [7]:
trials = remove_export_columns(df_trials)

for column in trials.select_dtypes(include=["object"]).columns:
    trials[column] = clean_text(trials[column])

trial_date_columns = [
    "Decision_date", "Start_date", "End_date", "Global_end_of_the_trial", "Last_updated",
]
for column in trial_date_columns:
    if column in trials:
        trials[column] = pd.to_datetime(trials[column], errors="coerce")

trials["Number_of_participants_enrolled"] = pd.to_numeric(
    trials["Number_of_participants_enrolled"], errors="coerce"
).astype("Int64")

before = len(trials)
trials = trials.drop_duplicates(subset=["Trial_number"], keep="first").reset_index(drop=True)
trial_countries = split_trial_locations(trials)

country_rollup = (
    trial_countries.groupby("Trial_number")["country_name"]
    .agg(lambda values: "; ".join(sorted(set(values))))
    .rename("country_names")
)
country_count = trial_countries.groupby("Trial_number")["country_name"].nunique().rename("country_count")
trials = trials.merge(country_rollup, on="Trial_number", how="left")
trials = trials.merge(country_count, on="Trial_number", how="left")
trials["country_count"] = trials["country_count"].fillna(0).astype("Int64")
trials["decision_year"] = trials["Decision_date"].dt.year.astype("Int64")

print(f"Trial rows: {before:,} -> {len(trials):,} ({before - len(trials):,} duplicate trial IDs removed)")
print(f"Trial–country relationships: {len(trial_countries):,}")
print(f"Trial countries: {trial_countries['country_name'].nunique():,}")
trial_countries.head(10)

/var/folders/_z/vq1q1pzj639b28jbhlxzd0zw0000gn/T/ipykernel_42773/668611286.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in trials.select_dtypes(include=["object"]).columns:


Trial rows: 5,918 -> 5,918 (0 duplicate trial IDs removed)
Trial–country relationships: 23,280
Trial countries: 31


,Trial_number,country_name,recruitment_status
0,2023-509723-41-00,Spain,Not authorised
1,2022-501417-31-00,Poland,Not authorised
2,2022-501417-31-00,Italy,Not authorised
3,2022-501417-31-00,Germany,Not authorised
4,2022-501417-31-00,Sweden,Not authorised
5,2022-501417-31-00,Austria,Not authorised
6,2022-501417-31-00,France,Not authorised
7,2022-501417-31-00,Spain,Not authorised
8,2022-501417-31-00,Ireland,Not authorised
9,2022-501417-31-00,Belgium,Not authorised


## 5. Data-quality checks

In [8]:
assert trials["Trial_number"].is_unique, "Trial_number must be unique after cleaning."
assert not trial_countries.duplicated(["Trial_number", "country_name"]).any(), "Duplicate trial-country pair found."
assert consur["projectID"].notna().all(), "CORDIS contains missing project IDs."

quality_report = pd.DataFrame([
    {
        "dataset": "CORDIS",
        "rows": len(consur),
        "unique_entities": consur["projectID"].nunique(),
        "missing_country": consur["country_name"].isna().sum(),
    },
    {
        "dataset": "CTIS trials",
        "rows": len(trials),
        "unique_entities": trials["Trial_number"].nunique(),
        "missing_country": trials["country_names"].isna().sum(),
    },
])
quality_report

,dataset,rows,unique_entities,missing_country
0,CORDIS,201690,36936,272
1,CTIS trials,5918,5918,0


## 6. Export dashboard-ready tables

In [10]:
consur.to_csv("../../_local_data_backup/ConsurDatabase_clean.csv")
trials.to_csv("../Scripts/data/data_final/TrialsDatabase_clean.csv")
trial_countries.to_csv("../Scripts/data/data_final/TrialCountry_clean.csv")